In [45]:
!pip install -q -U langchain langchain-google-genai langchain-core

In [46]:
import os
import json
from typing import Dict, Any, List

from langchain_core.tools import tool
from langchain_google_genai import ChatGoogleGenerativeAI


In [47]:
os.environ["GOOGLE_API_KEY"] = ""


In [48]:
symptom_category_keywords = {
    "respiratory": [
        "cough",
        "mild cough",
        "persistent cough",
        "breathing difficulty",
        "shortness of breath",
        "chest congestion"
    ],

    "cardiac": [
        "chest pain",
        "palpitations",
        "heart rate",
        "dizziness"
    ],

    "infection": [
        "fever",
        "high fever",
        "chills",
        "body ache",
        "fatigue"
    ],

    "neurological": [
        "severe headache",
        "confusion",
        "fainting",
        "weakness"
    ]
}

In [49]:
severity_weights = {
    "breathing difficulty": 30,
    "shortness of breath": 30,
    "chest pain": 35,
    "dizziness": 15,
    "palpitations": 20,
    "high fever": 20,
    "fever": 10,
    "persistent cough": 10,
    "mild cough": 5,
    "cough": 5,
    "chills": 10,
    "body ache": 5,
    "fatigue": 5,
    "severe headache": 15,
    "confusion": 30,
    "fainting": 35,
    "weakness": 15,
    "chest congestion": 10
}


In [50]:
age_risk_weights = {
    "young": 0,
    "adult": 0,
    "older_adult": 5
}


In [51]:
required_information = [
    "age",
    "symptom_duration",
    "symptoms"
]


In [52]:
def determine_risk_level(score: int) -> str:
    if score >= 60:
        return "HIGH"
    elif score >= 30:
        return "MODERATE"
    else:
        return "LOW"


In [53]:
def check_human_intervention(
    severity_score: int,
    matched_indicators: list
) -> bool:

    urgent_indicators = [
        "chest pain",
        "breathing difficulty",
        "shortness of breath",
        "confusion",
        "fainting"
    ]

    if severity_score >= 60:
        return True

    for indicator in matched_indicators:
        if indicator in urgent_indicators:
            return True

    return False


In [54]:
from langchain_core.tools import tool


In [55]:
@tool
def healthcare_assessment(
    symptoms: str,
    age: int = None,
    basic_information: str = ""
) -> dict:
    """
    Performs an initial synthetic healthcare triage assessment.

    Inputs:
    - symptoms: Patient's reported symptoms.
    - age: Patient's age, if known.
    - basic_information: Other relevant patient information.

    Returns:
    A structured assessment containing severity score,
    risk level, matched indicators, missing information,
    and whether human medical intervention is required.

    This tool does not provide a medical diagnosis.
    """


    symptoms_lower = symptoms.lower()



    matched_indicators = []

    for category, keywords in symptom_category_keywords.items():

        for keyword in keywords:

            if keyword.lower() in symptoms_lower:

                if keyword not in matched_indicators:
                    matched_indicators.append(keyword)



    severity_score = 0

    for indicator in matched_indicators:
        severity_score += severity_weights.get(indicator, 0)



    if age is not None and age >= 65:
        severity_score += 5


    severity_score = min(severity_score, 100)



    risk_level = determine_risk_level(severity_score)



    missing_indicators = []

    if not symptoms.strip():
        missing_indicators.append("symptoms")

    if age is None:
        missing_indicators.append("age")

    if not basic_information.strip():
        missing_indicators.append("basic patient information")



    human_intervention_required = check_human_intervention(
        severity_score,
        matched_indicators
    )



    return {
        "severity_score": severity_score,
        "risk_level": risk_level,
        "matched_indicators": matched_indicators,
        "missing_indicators": missing_indicators,
        "human_intervention_required": human_intervention_required
    }


In [56]:
llm = ChatGoogleGenerativeAI(
   model="gemini-3.6-flash"
)

In [57]:
system_prompt = """
You are a Healthcare Patient Triage Assistant.

Your purpose is to support an initial patient urgency-assessment
workflow using a synthetic healthcare assessment tool.

IMPORTANT:
You are NOT a doctor.
You must NOT provide a definitive medical diagnosis.
You must NOT claim that a patient has a specific disease or condition.

TOOL USAGE:
1. Use the healthcare_assessment tool whenever the user provides
   symptoms or requests a structured assessment of a patient's
   urgency or risk.
2. Do not manually calculate the severity score when the tool can
   perform the assessment.
3. Use the structured result returned by the tool as the basis
   for your response.

PATIENT INFORMATION:
4. Base your assessment only on information provided by the user.
5. Never invent missing information such as age, symptoms,
   duration, medical history, or vital signs.
6. If important information is missing, clearly identify it.
7. Do not assume that an unknown symptom or missing value is normal.

RESPONSE STYLE:
8. Keep responses short and structured.
9. Clearly state the risk level returned by the tool.
10. Mention the important matched risk indicators.
11. Mention important missing information when applicable.
12. Provide a concise recommendation based on the tool output.

HUMAN MEDICAL ATTENTION:
13. If the tool indicates that human medical intervention is required,
    highlight this immediately and prominently.
14. Do not reassure a user that a potentially urgent situation is safe.
15. If the situation appears urgent according to the tool, recommend
    prompt human medical assessment.

UNKNOWN CATEGORIES:
16. If the tool returns UNKNOWN because the symptoms do not match
    the synthetic categories, explain that the system cannot make
    a reliable automated assessment for those symptoms.
17. Do not invent a diagnosis or risk classification for unsupported
    symptoms.

DISCLAIMER:
18. Clearly communicate that this is an initial synthetic triage
    assessment and not a medical diagnosis.

Do not expose internal reasoning or hidden chain-of-thought.
Provide only the relevant assessment and recommendation.
"""


In [58]:
tools = [
    healthcare_assessment
]


In [59]:
print(tools)


[StructuredTool(name='healthcare_assessment', description="Performs an initial synthetic healthcare triage assessment.\n\nInputs:\n- symptoms: Patient's reported symptoms.\n- age: Patient's age, if known.\n- basic_information: Other relevant patient information.\n\nReturns:\nA structured assessment containing severity score,\nrisk level, matched indicators, missing information,\nand whether human medical intervention is required.\n\nThis tool does not provide a medical diagnosis.", args_schema=<class 'langchain_core.utils.pydantic.healthcare_assessment'>, func=<function healthcare_assessment at 0x7c45fc137560>)]


In [60]:
from langchain.agents import create_agent


In [61]:
agent = create_agent(
    model=llm,
    tools=tools
)


In [62]:
from langchain_core.messages import SystemMessage, HumanMessage


In [63]:
def run_agent(user_prompt: str) -> str:
    result = agent.invoke(
        {
            "messages": [
                SystemMessage(content=system_prompt),
                HumanMessage(content=user_prompt)
            ]
        }
    )

    return result["messages"][-1].content


In [64]:
patient_query = """
Patient is 62 years old and is experiencing chest pain,
breathing difficulty and dizziness since this morning.
"""

In [65]:
response = run_agent(patient_query)

print(response)

[{'type': 'text', 'text': '**URGENT: HUMAN MEDICAL INTERVENTION IS REQUIRED IMMEDIATELY.**\n\n### Triage Assessment Summary\n* **Risk Level:** HIGH\n* **Severity Score:** 80 / 100\n* **Matched Risk Indicators:** Chest pain, breathing difficulty, dizziness\n\n### Missing Information\n* Past medical history, current vital signs, and medication history.\n\n### Recommendation\nSeek **immediate emergency medical evaluation** (e.g., call emergency services / 911 or go to the nearest emergency room). Do not drive yourself to the hospital.\n\n---\n*Disclaimer: This is an initial synthetic triage assessment and is NOT a medical diagnosis or a substitute for professional medical care.*', 'extras': {'signature': 'EvsKCvgKARFNMg8l8z9QmSwVbSKt1fimoV4vsWb/sehYZogvzo0ITd1nmOOfvCvaw8GD8thwGkA83zl+mEYlerJF58+FqR6k3O1yF8BMI8AHaDwBPXaLLDL7rIpNdH1K/D4QaGJ343+QSjwnqSkgy5ckeSxjL0lp2c+ZXQbBycMUIWrJ9uMXxf7KdtvZ6s8ps57prO1yWZU5Dgxv3xWA6iVqLvB0Rtw1rE/LNoo3fCBpbotiSWP/jBbj8UAQW9exLFeVgGoHRC3ngVQCOYzlr80ag2ewBkjW

In [66]:
test_cases = [

    {
        "name": "Test Case 1 - Low Risk Scenario",
        "prompt": """
        Patient is 25 years old and has had a mild cough
        for two days with no breathing difficulty or chest pain.
        """
    },


    {
        "name": "Test Case 2 - Moderate Risk Scenario",
        "prompt": """
        Patient is 48 years old and has fever, fatigue and
        persistent cough for four days.
        """
    },


    {
        "name": "Test Case 3 - High Risk Scenario",
        "prompt": """
        Patient is 67 years old and is experiencing chest pain,
        breathing difficulty and dizziness since this morning.
        """
    }

]


In [67]:
for test in test_cases:


    print(test["name"])


    response = run_agent(test["prompt"])

    print(response)
    print("\n")


Test Case 1 - Low Risk Scenario
[{'type': 'text', 'text': '### Triage Assessment Summary\n\n* **Risk Level:** LOW (Severity Score: 10/100)\n* **Human Medical Intervention Required:** No (based on current automated assessment)\n* **Matched Risk Indicators:** Mild cough, Cough\n\n---\n\n### Patient Information & Missing Data\n* **Provided Information:** Age 25, mild cough for 2 days, no chest pain, no breathing difficulty.\n* **Missing Information:** Medical history, presence of fever, or other associated symptoms.\n\n---\n\n### Recommendation\n* **Next Steps:** Self-monitor symptoms at home. Stay hydrated and rest.\n* **When to Seek Care:** If symptoms worsen, or if new symptoms develop—such as shortness of breath, high fever, or chest tightness—seek prompt evaluation by a medical professional.\n\n---\n\n*Disclaimer: This is an initial synthetic triage assessment and not a medical diagnosis. Always consult a qualified healthcare provider for proper medical evaluation and advice.*', 'ext

In [68]:
extended_test_cases = [
    {
        "prompt": """
        Patient is 35 years old and has skin itching
        for three days.
        """
    },

    {
        "prompt": """
        Patient has cough and chest congestion.
        """
    },

    {
        "prompt": """
        Patient is 70 years old and has chest pain,
        breathing difficulty, fever, dizziness and fatigue.
        """
    }
]


for i in extended_test_cases:
    print(run_agent(i["prompt"]))
    print("\n" + "="*80 + "\n")


[{'type': 'text', 'text': '**Initial Triage Assessment**\n\n* **Risk Level:** LOW (Severity Score: 0/10)\n* **Human Medical Intervention Required:** No immediate emergency intervention indicated based on provided details.\n* **Matched Risk Indicators:** None\n* **Important Missing Information:** Rash characteristics, affected body area/extent, presence of swelling or breathing difficulty, recent medication/allergy history.\n\n---\n\n### **Recommendation**\n* Monitor symptoms for changes, spreading, or worsening.\n* Seek evaluation from a qualified healthcare provider or dermatologist if itching persists, worsens, or is accompanied by rash, pain, or systemic symptoms (e.g., fever, swelling).\n\n---\n\n*Disclaimer: This is an initial synthetic triage assessment for informational purposes and is NOT a medical diagnosis. Always consult a healthcare professional for clinical evaluation and medical advice.*', 'extras': {'signature': 'Et4FCtsFARFNMg90mIznSzageYhA9jxvXyzkLHfciPIOkzJ5Ua40ROwV3+